# Step 2 – From detectors to measurement locations

In step 1 we got data for **32 detectors**. But we want to analyse **12 locations**
(6 eastbound, 6 westbound). Some locations have several detectors:

- **MONICA** detectors measure **one lane** each.
- **MONIBAS** detectors measure **all lanes** of the carriageway together.

If a location has both, adding them up would count the same cars twice.
In this notebook we check which detectors measure the same traffic, choose one source per
location, and make one table with one row per location, date and hour.

In [1]:
import pandas as pd

data = pd.read_csv("../processed_data/A1_detectors_hourly.csv")

# Shorter detector names are easier to read
data["short"] = data["detector"].str.replace("RWS01_", "").str.replace("MONICA_", "").str.replace("MONIBAS_", "")
data.head()

,detector,date,hour,intensity,speed,short
0,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,0,265.3,115,0011hrl0434ra
1,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,1,180.0,107,0011hrl0434ra
2,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,2,148.6,106,0011hrl0434ra
3,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,3,142.9,104,0011hrl0434ra
4,RWS01_MONIBAS_0011hrl0434ra,2026-08-25,4,290.0,104,0011hrl0434ra


## 2.1 Do the lane detectors add up to the MONIBAS detector?

We compare the sum of the lane detectors with the MONIBAS detector at the same location,
for a few hours on a normal Tuesday (8 September).

In [2]:
day = data[data["date"] == "2026-09-08"]
table = day.pivot_table(index="hour", columns="short", values="intensity")

def compare_lanes(lanes, monibas, hours=[3, 7, 8, 12, 17]):
    """Show the lane detectors, their sum and the MONIBAS detector side by side."""
    t = table[lanes].copy()
    t["sum of lanes"] = t[lanes].sum(axis=1)
    t["MONIBAS"] = table[monibas]
    return t.loc[hours]

In [3]:
# Re 90.5 (eastbound, after N345): 5 lanes and one MONIBAS detector
compare_lanes(["00D0010E243CD0070007", "00D0010E243CD007000B", "00D0010E243CD007000F",
               "00D0010E243CD0070013", "00D0010E243CD0070017"], "0011hrr0905ra")

short,00D0010E243CD0070007,00D0010E243CD007000B,00D0010E243CD007000F,00D0010E243CD0070013,00D0010E243CD0070017,sum of lanes,MONIBAS
hour,,,,,,,
3,0.0,12.0,53.0,45.0,90.0,200.0,200.0
7,116.0,775.0,1020.0,997.0,899.0,3807.0,3826.0
8,103.0,796.0,1069.0,995.0,945.0,3908.0,3908.0
12,107.0,826.0,927.0,788.0,778.0,3426.0,3426.0
17,334.6,1378.0,1484.7,1252.9,1068.8,5519.0,5519.0


At Re 90.5 the sum of the 5 lanes is (almost) exactly the MONIBAS value: **they measure the same cars**.

In [4]:
# Li 90.6 (westbound, before N345): lanes 0005, 0009, 000D and two extra lanes 0085, 0089
compare_lanes(["00D0010E2432D0050005", "00D0010E2432D0050009", "00D0010E2432D005000D"], "0011hrl0905ra")

short,00D0010E2432D0050005,00D0010E2432D0050009,00D0010E2432D005000D,sum of lanes,MONIBAS
hour,,,,,
3,2.0,29.0,141.0,172.0,172.0
7,643.0,1255.0,868.0,2766.0,2766.0
8,446.0,1133.0,869.0,2448.0,2448.0
12,123.0,664.0,755.0,1542.0,1542.0
17,155.6,624.4,753.6,1533.6,1533.6


In [5]:
# The extra lanes 0085 and 0089 are NOT included in the MONIBAS value
table[["00D0010E2432D0050085", "00D0010E2432D0050089"]].loc[[3, 7, 8, 12, 17]]

short,00D0010E2432D0050085,00D0010E2432D0050089
hour,,
3,35.0,114.0
7,1463.0,1615.0
8,1266.0,1490.0
12,645.0,982.0
17,818.6,1272.2


At Li 90.6 the MONIBAS value equals lanes 0005 + 0009 + 000D. Lanes 0085 and 0089 carry a lot of
traffic but are not part of it, so they probably belong to another carriageway.
We leave them out.

In [6]:
# Re 86.5 (eastbound, before Beekbergen): 5 lanes, compared with MONIBAS 0011hrr0851ra_1 (at km 85.1)
compare_lanes(["00D0010D840AD0070007", "00D0010D840AD007000B", "00D0010D840AD007000F",
               "00D0010D840AD0070087", "00D0010D840AD007008B"], "0011hrr0851ra_1")

short,00D0010D840AD0070007,00D0010D840AD007000B,00D0010D840AD007000F,00D0010D840AD0070087,00D0010D840AD007008B,sum of lanes,MONIBAS
hour,,,,,,,
3,0.0,23.0,69.0,20.0,12.0,124.0,124.0
7,264.0,684.0,651.0,785.0,108.0,2492.0,2542.4
8,207.0,701.0,667.0,685.0,138.0,2398.0,2407.0
12,249.0,775.0,654.0,673.0,202.0,2553.0,2530.0
17,560.3,1092.2,788.1,797.3,218.6,3456.5,3438.3


At Re 86.5 the sum of all 5 lanes is within about 2% of the MONIBAS detector 1.4 km earlier,
so they measure the same traffic. We use the MONIBAS detector.

At **Re 43.6** and **Re 46.4** there is no MONIBAS detector. There we add up the two main lanes
(0007 and 000B). The other IDs at these locations (0087/008B at 43.6, 0187 at 46.4) are extra lanes
that we leave out. *To do: check on the Dexter map which lanes these are.*

## 2.2 Our choice per location

| Location | Direction | Detector(s) used |
|---|---|---|
| Re 43.6 | eastbound | lanes 0007 + 000B |
| Re 46.4 | eastbound | lanes 0007 + 000B |
| Re 86.5 | eastbound | MONIBAS 0011hrr0851ra_1 |
| Re 90.5 | eastbound | MONIBAS 0011hrr0905ra |
| Re 104.4 | eastbound | MONIBAS 0011hrr1044ra |
| Re 107.7 | eastbound | MONIBAS 0011hrr1076ra |
| Li 107.6 | westbound | MONIBAS 0011hrl1076ra |
| Li 98.7 | westbound | MONIBAS 0011hrl0987ra |
| Li 90.6 | westbound | MONIBAS 0011hrl0905ra |
| Li 85.6 | westbound | MONIBAS 0011hrl0856ra |
| Li 46.0 | westbound | MONIBAS 0011hrl0460ra |
| Li 43.4 | westbound | MONIBAS 0011hrl0434ra |

In [7]:
locations = {
    "Re 43.6":  ["00D00106CC55D0070007", "00D00106CC55D007000B"],
    "Re 46.4":  ["10D00107400FD0070007", "10D00107400FD007000B"],
    "Re 86.5":  ["0011hrr0851ra_1"],
    "Re 90.5":  ["0011hrr0905ra"],
    "Re 104.4": ["0011hrr1044ra"],
    "Re 107.7": ["0011hrr1076ra"],
    "Li 107.6": ["0011hrl1076ra"],
    "Li 98.7":  ["0011hrl0987ra"],
    "Li 90.6":  ["0011hrl0905ra"],
    "Li 85.6":  ["0011hrl0856ra"],
    "Li 46.0":  ["0011hrl0460ra"],
    "Li 43.4":  ["0011hrl0434ra"],
}

## 2.3 Combine the detectors per location

- **Intensity**: we add up the detectors. If one of the lanes is missing for an hour,
  we set that hour to missing (otherwise the total would look like a drop in traffic).
- **Speed**: we take the average speed of the lanes, weighted by the number of cars per lane.
  A lane with more cars counts more. Speed 0 means "no cars measured" (see step 1), so we ignore it.

In [8]:
data.loc[data["speed"] == 0, "speed"] = None   # speed 0 = no data

rows = []
for location, detectors in locations.items():
    sub = data[data["short"].isin(detectors)]
    for (date, hour), group in sub.groupby(["date", "hour"]):
        if len(group) < len(detectors):
            intensity, speed = None, None          # a lane is missing this hour
        else:
            intensity = group["intensity"].sum()
            valid = group.dropna(subset=["speed"])
            if valid["intensity"].sum() > 0:
                speed = (valid["speed"] * valid["intensity"]).sum() / valid["intensity"].sum()
            else:
                speed = None
        rows.append([location, date, hour, intensity, speed])

locations_data = pd.DataFrame(rows, columns=["location", "date", "hour", "intensity", "speed"])
locations_data["direction"] = locations_data["location"].str[:2].map({"Re": "eastbound", "Li": "westbound"})
locations_data.head()

,location,date,hour,intensity,speed,direction
0,Re 43.6,2026-08-25,0,180.0,111.633333,eastbound
1,Re 43.6,2026-08-25,1,119.0,108.924370,eastbound
2,Re 43.6,2026-08-25,2,109.0,108.330275,eastbound
3,Re 43.6,2026-08-25,3,102.0,107.352941,eastbound
4,Re 43.6,2026-08-25,4,138.0,107.507246,eastbound


## 2.4 Check

In [9]:
# 12 locations x 7 days x 24 hours = 2016 rows (a few hours can be missing)
print(len(locations_data))
locations_data.groupby("location").size()


2012


location
Li 107.6    168
Li 43.4     168
Li 46.0     168
Li 85.6     168
Li 90.6     168
Li 98.7     168
Re 104.4    168
Re 107.7    168
Re 43.6     164
Re 46.4     168
Re 86.5     168
Re 90.5     168
dtype: int64

In [10]:
# Quick look: total cars per location per day
locations_data.pivot_table(index="location", columns="date", values="intensity", aggfunc="sum").round(0)

date,2026-08-25,2026-09-01,2026-09-08,2026-09-15,2026-09-22,2026-09-29,2026-10-06
location,,,,,,,
Li 107.6,42323.0,42743.0,41841.0,43496.0,42678.0,43560.0,43025.0
Li 43.4,54382.0,54237.0,47168.0,50566.0,55706.0,56534.0,52514.0
Li 46.0,49943.0,50353.0,49911.0,45509.0,52221.0,51865.0,47712.0
Li 85.6,57600.0,56375.0,57600.0,54905.0,57600.0,57600.0,57600.0
Li 90.6,29333.0,26100.0,25885.0,25848.0,27332.0,27215.0,26721.0
Li 98.7,55350.0,56745.0,55215.0,57604.0,56728.0,57979.0,57834.0
Re 104.4,53951.0,54608.0,54770.0,56714.0,55539.0,56620.0,56375.0
Re 107.7,41647.0,41398.0,41938.0,43220.0,42059.0,42834.0,42367.0
Re 43.6,30425.0,30459.0,28077.0,31060.0,29919.0,32401.0,29587.0


In the table above **Li 85.6** has exactly 57,600 vehicles on four days. That is suspicious:
it would mean exactly 2,400 vehicles in every hour of the day. Let's look at the hourly values.

In [11]:
locations_data[locations_data["location"] == "Li 85.6"].pivot(index="hour", columns="date", values="intensity")

date,2026-08-25,2026-09-01,2026-09-08,2026-09-15,2026-09-22,2026-09-29,2026-10-06
hour,,,,,,,
0,2400.0,2400.0,2400.0,2100.0,2400.0,2400.0,2400.0
1,2400.0,2400.0,2400.0,2100.0,2400.0,2400.0,2400.0
2,2400.0,2400.0,2400.0,2100.0,2400.0,2400.0,2400.0
3,2400.0,2400.0,2400.0,2100.0,2400.0,2400.0,2400.0
4,2400.0,2400.0,2400.0,2100.0,2400.0,2400.0,2400.0
5,2400.0,2400.0,2400.0,2395.0,2400.0,2400.0,2400.0
6,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0
7,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0
8,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0,2400.0


In [12]:
# Check all locations: how often does the most common intensity value occur?
# Real traffic counts almost never give exactly the same number many times.
for location, group in locations_data.groupby("location"):
    most_common = group["intensity"].value_counts().iloc[0]
    print(location, "- most common value occurs", most_common, "times")

Li 107.6 - most common value occurs 2 times
Li 43.4 - most common value occurs 2 times
Li 46.0 - most common value occurs 2 times
Li 85.6 - most common value occurs 153 times
Li 90.6 - most common value occurs 2 times
Li 98.7 - most common value occurs 2 times
Re 104.4 - most common value occurs 2 times
Re 107.7 - most common value occurs 2 times
Re 43.6 - most common value occurs 2 times
Re 46.4 - most common value occurs 2 times
Re 86.5 - most common value occurs 2 times
Re 90.5 - most common value occurs 3 times


Li 85.6 shows 2,400 vehicles/hour and 106 km/h in almost every hour, also at night.
This is not a real measurement but a fixed replacement value, probably because the detector
was broken. **We remove Li 85.6 from the analysis.** All other locations look normal.

In [15]:
locations_data = locations_data[locations_data["location"] != "Li 85.6"]
print(locations_data["location"].nunique(), "locations left")

11 locations left


**Note on missing hours:** Re 43.6 has 164 instead of 168 hours, because for a few hours
both lanes were missing in the Dexter export (see step 1). These hours are simply not in the table.

## 2.5 Save the result

In [16]:
locations_data.to_csv("../processed_data/A1_locations_hourly.csv", index=False)
print("Saved", len(locations_data), "rows")

Saved 1844 rows
